In [1]:
import random
import numpy as np
import torch

SID4 = 2733
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"SID4: {SID4} | SEED: {SEED} | SLICE: {SLICE} | HP_ID: {HP_ID} | CLS_A: {CLS_A} | CLS_B: {CLS_B}")

SID4: 2733 | SEED: 2733 | SLICE: 733 | HP_ID: 3 | CLS_A: 3 | CLS_B: 7


In [2]:
from datasets import load_dataset

#  the dialogsum dataset
dataset = load_dataset("neil-code/dialogsum-test")

# 500 rows starting from my slice index
subset_size = 500
train_subset = dataset["train"].select(range(SLICE, min(SLICE + subset_size, len(dataset["train"]))))

# map the raw columns to simple input/target keys
def format_dataset(sample):
    return {
        "input": sample["dialogue"],
        "target": sample["summary"]
    }

# applying the formatting to the subset
processed_data = train_subset.map(format_dataset)


for i in range(2):
    print(f"=== Sample {i+1} (Dataset Index {SLICE + i}) ===")
    print(f"Input:  {processed_data[i]['input']}")
    print(f"Target: {processed_data[i]['target']}\n")

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/441k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/499 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

=== Sample 1 (Dataset Index 733) ===
Input:  #Person1#: I've heard that you are going to call for a tender. What kind of goods would you like to buy?
#Person2#: Yes, we're ready to start the invitation to tender. You're well informed. We'd like to build new factory.
#Person1#: When do you open the tender? And where?
#Person2#: We intend to open the tender at first next month in Beijing.
#Person1#: And when is the closing date?
#Person2#: The time period is set on June 29th.
#Person1#: Could you please tell me something more about the conditions for the tender?
#Person2#: All right. The area of the factory buildings is around 8, 000 square meters. You can read the details in the invitations which will be sent tomorrow.
#Person1#: Thank you for your information.
#Person2#: You're welcome.
#Person1#: I'm sorry to have taken up too much of your time.
#Person2#: It doesn't matter. We'll be in touch.
#Person1#: Okay, bye!
#Person2#: Bye!
Target: #Person2# calls for a tender to build a new fa

In [3]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# the pretrained model and tokenizer
model_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)


baseline_outputs = []
print("=== BASELINE INFERENCE (Before Fine-Tuning) ===")

for i in range(2):
    dialogue = processed_data[i]["input"]

    # turn the dialogue text into tokens
    inputs = tokenizer(dialogue, return_tensors="pt", max_length=512, truncation=True)

    # generate a summary without updating model weights
    with torch.no_grad():
        outputs = baseline_model.generate(**inputs, max_new_tokens=64)

    # decode the token ids back to readable text
    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    baseline_outputs.append(decoded_output)

    print(f"\nSample {i+1} Baseline Output:")
    print(decoded_output)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

=== BASELINE INFERENCE (Before Fine-Tuning) ===

Sample 1 Baseline Output:
#Person1#: Thank you for your information.

Sample 2 Baseline Output:
#Person1#: Thanks for calling, Blanca.


In [5]:

!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 19.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [6]:
from peft import LoraConfig, TaskType, get_peft_model

# set up the lora configuration parameters
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=4,
    lora_alpha=32,  # scaling parameter
    lora_dropout=0.05 # dropout probability for regularization
)

# the baseline model with the peft lora configuration
peft_model = get_peft_model(baseline_model, peft_config)


peft_model.print_trainable_parameters()

trainable params: 172,032 || all params: 77,133,184 || trainable%: 0.2230


In [10]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# Tokenize dialogues and set target summaries as labels
def preprocess_function(examples):
    model_inputs = tokenizer(examples["input"], max_length=512, truncation=True)
    labels = tokenizer(examples["target"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply tokenization to the dataset
tokenized_dataset = processed_data.map(preprocess_function, batched=True)

# Data collator handles dynamic padding for batched sequence-to-sequence training
data_collator = DataCollatorForSeq2Seq(tokenizer, model=peft_model)

# Define training parameters, using lower learning rate for HP_ID 3
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-4,          # Lower LR for arm 3
    per_device_train_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    seed=SEED
)

# Initialize the trainer with the data collator included
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)


trainer.train()


peft_model.save_pretrained("./lora_dialogsum_checkpoint")

Step,Training Loss
10,2.567105
20,2.403513
30,2.291462
40,2.172524
50,2.073964
60,1.979379
70,2.006142
80,1.900265
90,1.845356
100,1.929589


In [11]:

peft_model.eval()

print("=== POST-FINE-TUNING INFERENCE (After LoRA) ===")
for i in range(2):
    dialogue = processed_data[i]["input"]

    # tokenize the input dialogue
    inputs = tokenizer(dialogue, return_tensors="pt", max_length=512, truncation=True)


    with torch.no_grad():
        outputs = peft_model.generate(**inputs, max_new_tokens=64)


    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"\nSample {i+1} Fine-Tuned Output:")
    print(decoded_output)

=== POST-FINE-TUNING INFERENCE (After LoRA) ===

Sample 1 Fine-Tuned Output:
#Person1# is going to call for a tender. #Person2# is going to open the tender at first next month in Beijing. #Person1# will open the tender at first next month in Beijing. #Person2# will open the tender at first next month in Beijing.

Sample 2 Fine-Tuned Output:
#Person1# is interested in one-bedroom apartment. #Person2# is interested in one-bedroom apartment. #Person1# will come over tomorrow to take a look. #Person2# will see #Person1# tomorrow.


In [12]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

 Load a fresh base model
model_r16 = AutoModelForSeq2SeqLM.from_pretrained(model_id)

 Configure LoRA with r = 16
peft_config_r16 = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05
)

 Attach adapters and check parameter count
peft_model_r16 = get_peft_model(model_r16, peft_config_r16)
peft_model_r16.print_trainable_parameters()

 Set up training arguments
training_args_r16 = TrainingArguments(
    output_dir="./results_r16",
    learning_rate=1e-4,
    per_device_train_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    seed=SEED
)
 Initialize trainer and train
trainer_r16 = Trainer(
    model=peft_model_r16,
    args=training_args_r16,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer_r16.train()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


Step,Training Loss
10,2.570209
20,2.417581
30,2.305521
40,2.186996
50,2.086524
60,1.982259
70,2.008565
80,1.910417
90,1.847049
100,1.930986


TrainOutput(global_step=125, training_loss=2.071483108520508, metrics={'train_runtime': 709.878, 'train_samples_per_second': 0.704, 'train_steps_per_second': 0.176, 'total_flos': 60753339064320.0, 'train_loss': 2.071483108520508, 'epoch': 1.0})